**para el dataser https://github.com/owid/covid-19-data/blob/master/public/data/owid-covid-data.csv**

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings

# Ocultar advertencias para mantener el notebook limpio
warnings.filterwarnings("ignore")

# Crear directorios necesarios si no existen
os.makedirs("../data", exist_ok=True)
os.makedirs("../results", exist_ok=True)

In [2]:
import requests

url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

response = requests.get(url)

if response.status_code == 200:
    with open("../data/owid-covid-data.csv", "wb") as f:
        f.write(response.content)
    print("Archivo descargado exitosamente en '../data/owid-covid-data.csv'")
else:
    print(f"Error al descargar el archivo. Código de estado: {response.status_code}")

Archivo descargado exitosamente en '../data/owid-covid-data.csv'


In [3]:
df = pd.read_csv(
    "../data/owid-covid-data.csv",
    usecols=[
        "location",
        "date",
        "total_cases",
        "new_cases",
        "total_deaths",
        "population",
    ],
)
df["date"] = pd.to_datetime(df["date"])
df.head()

,location,date,total_cases,new_cases,total_deaths,population
0,Afghanistan,2020-01-05,0.0,0.0,0.0,41128772
1,Afghanistan,2020-01-06,0.0,0.0,0.0,41128772
2,Afghanistan,2020-01-07,0.0,0.0,0.0,41128772
3,Afghanistan,2020-01-08,0.0,0.0,0.0,41128772
4,Afghanistan,2020-01-09,0.0,0.0,0.0,41128772


In [4]:
# ¿Cuántos valores nulos hay?
print("Valores nulos por columna:")
display(df.isnull().sum())

# Rango de fechas
print("Rango de fechas:")
display(df["date"].min(), df["date"].max())

# informacion de las columnas
print("Información de las columnas:")
display(df.info())

# cantoidad de paises
print("Cantidad de países únicos:")
display(len(df["location"].unique()))

Valores nulos por columna:


location            0
date                0
total_cases     17631
new_cases       19276
total_deaths    17631
population          0
dtype: int64

Rango de fechas:


Timestamp('2020-01-01 00:00:00')

Timestamp('2024-08-14 00:00:00')

Información de las columnas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429435 entries, 0 to 429434
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   location      429435 non-null  object        
 1   date          429435 non-null  datetime64[ns]
 2   total_cases   411804 non-null  float64       
 3   new_cases     410159 non-null  float64       
 4   total_deaths  411804 non-null  float64       
 5   population    429435 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 19.7+ MB


None

Cantidad de países únicos:


255

In [5]:
df_clean = df.copy()
# Rellenar con el anterior para total_cases y total_deaths
df_clean["total_cases"] = df_clean.groupby("location")["total_cases"].ffill().fillna(0)
df_clean["total_deaths"] = (
    df_clean.groupby("location")["total_deaths"].ffill().fillna(0)
)

# Rellenar con cero para new_cases
df_clean["new_cases"] = df_clean["new_cases"].fillna(0)

# Verificar
print(df_clean.isnull().sum())

location        0
date            0
total_cases     0
new_cases       0
total_deaths    0
population      0
dtype: int64


In [9]:
no_son_paises = [
    "World",
    "High income",
    "Upper middle income",
    "Europe",
    "Asia",
    "North America",
    "South America",
    "Lower middle income",
    "European Union",
    "Africa",
    "Low income",
    "Oceania",
]

In [28]:
# DataFrame solo con países (para ver el Top 10 de países, etc.)
df_paises = df_clean[~df_clean["location"].isin(no_son_paises)]

# DataFrame solo con regiones (para comparar continentes o niveles de ingreso)
df_regiones = df_clean[df_clean["location"].isin(no_son_paises)]

In [29]:
total_global = df_clean["new_cases"].sum()
total_global_pais = df_paises["new_cases"].sum()
total_global_region = df_regiones["new_cases"].sum()
resta_region = total_global - total_global_region

print(total_global)
print(total_global_pais)
print(total_global_region)
print(resta_region)
print(resta_region == total_global_pais)

3288392333.0
1736522219.0
1551870114.0
1736522219.0
True


In [35]:
# Caso por millon
df_paises["total_cases_per_million"] = (
    df_paises["total_cases"] / df_paises["population"]
) * 1_000_000

# Muertes por millón
df_paises["total_deaths_per_million"] = (
    df_paises["total_deaths"] / df_paises["population"]
) * 1_000_000

# Tasa de mortalidad
df_paises["fatality_rate"] = (
    df_paises["total_deaths"] / df_paises["total_cases"]
) * 100

df_paises.head()

,location,date,total_cases,new_cases,total_deaths,population,total_cases_per_million,total_deaths_per_million,fatality_rate
0,Afghanistan,2020-01-05,0.0,0.0,0.0,41128772,0.0,0.0,NaN
1,Afghanistan,2020-01-06,0.0,0.0,0.0,41128772,0.0,0.0,NaN
2,Afghanistan,2020-01-07,0.0,0.0,0.0,41128772,0.0,0.0,NaN
3,Afghanistan,2020-01-08,0.0,0.0,0.0,41128772,0.0,0.0,NaN
4,Afghanistan,2020-01-09,0.0,0.0,0.0,41128772,0.0,0.0,NaN


In [37]:
# Top 10 países por muerte
top_10_deaths = (
    df_paises.groupby("location")["total_deaths_per_million"]
    .last()
    .sort_values(ascending=False)
    .head(10)
)
top_10_deaths

location
Peru                      6489.799524
Bulgaria                  5706.319196
Bosnia and Herzegovina    5069.382378
Hungary                   4921.390980
North Macedonia           4765.939723
Slovenia                  4756.484325
Croatia                   4652.684958
Georgia                   4580.191407
Montenegro                4232.301358
Czechia                   4146.087427
Name: total_deaths_per_million, dtype: float64